# Fishing Problem with AMPL in Python
## AMPL Modeling for Book Problem 2.3


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```

The first call to `ampl_notebook(modules=["highs"], license_uuid="default")` may download the solver module if it is not already available.


In [14]:
from __future__ import annotations

from functools import wraps
from time import perf_counter


In [15]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
def create_ampl_instance(solver: str = "highs"):
    ampl = ampl_notebook(modules=[solver], license_uuid="default")
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl


def extract_solution(variable, variable_order):
    values = variable.get_values().to_dict()
    solution = {}

    for name in variable_order:
        if name in values:
            solution[name] = int(round(values[name]))
        else:
            solution[name] = int(round(values[(name,)]))

    return solution


def extract_matrix_solution(variable):
    values = variable.get_values().to_dict()
    solution = {}

    for key, value in values.items():
        if abs(value) > 1e-9:
            solution[key] = int(round(value))

    return solution


## Problem 1: Base Fishing Problem

Variables:
- `pacific_trips`
- `atlantic_trips`

Objective:
- minimize operating cost


In [17]:
BASE_EXPECTED = {
    "pacific_trips": 8,
    "atlantic_trips": 3,
    "cost": 134000,
}


@timed
def solve_base_fishing_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        var pacific_trips >= 0 integer;
        var atlantic_trips >= 0 integer;

        minimize Cost:
            13000 * pacific_trips + 10000 * atlantic_trips;

        subject to BigEye:
            150 * pacific_trips + 100 * atlantic_trips >= 1500;

        subject to Yellowfin:
            100 * pacific_trips + 200 * atlantic_trips >= 600;

        subject to Skipjack:
            50 * pacific_trips + 100 * atlantic_trips >= 700;
        '''
    )
    ampl.solve(solver=solver)

    solution = {
        "pacific_trips": int(round(ampl.get_value("pacific_trips"))),
        "atlantic_trips": int(round(ampl.get_value("atlantic_trips"))),
        "cost": int(round(ampl.get_value("Cost"))),
    }
    return solution


In [18]:
base_result, base_time = solve_base_fishing_with_ampl()

print("=== BASE FISHING RESULTS WITH AMPL ===")
print(f"Solution -> {base_result}")
print(f"Time     -> {base_time:.8f} seconds")

assert base_result == BASE_EXPECTED
print("The AMPL solution matches the textbook optimum.")


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: === BASE FISHING RESULTS WITH AMPL ===
Solution -> {'pacific_trips': 8, 'atlantic_trips': 3, 'cost': 134000}
Time     -> 1.55548746 seconds
The AMPL solution matches the textbook optimum.


## Problem 2: Selective-Fishing Variant

The book asks for a reformulation with decision variables of the form `Trips[ocean, species]`.

To keep the notebook executable, we use the same explicit assumption as in the pure Python notebook:
- Pacific selective trip: `300` tons of the targeted species
- Atlantic selective trip: `400` tons of the targeted species


In [19]:
OCEANS = ["Pacific", "Atlantic"]
SPECIES = ["Big Eye", "Yellowfin", "Skipjack"]
TRIP_COST = {"Pacific": 13000, "Atlantic": 10000}
SELECTIVE_HAUL = {"Pacific": 300, "Atlantic": 400}
DEMAND = {"Big Eye": 1500, "Yellowfin": 600, "Skipjack": 700}

VARIANT_EXPECTED = {
    ("Atlantic", "Big Eye"): 4,
    ("Atlantic", "Yellowfin"): 2,
    ("Atlantic", "Skipjack"): 2,
    "total_cost": 80000,
}


@timed
def solve_selective_fishing_variant_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set OCEANS ordered;
        set SPECIES ordered;

        param trip_cost {OCEANS} >= 0;
        param selective_haul {OCEANS} >= 0;
        param demand {SPECIES} >= 0;

        var Trips {o in OCEANS, s in SPECIES} integer >= 0;

        minimize TotalCost:
            sum {o in OCEANS, s in SPECIES} trip_cost[o] * Trips[o, s];

        subject to MeetDemand {s in SPECIES}:
            sum {o in OCEANS} selective_haul[o] * Trips[o, s] >= demand[s];
        '''
    )

    ampl.set["OCEANS"] = OCEANS
    ampl.set["SPECIES"] = SPECIES
    ampl.param["trip_cost"] = TRIP_COST
    ampl.param["selective_haul"] = SELECTIVE_HAUL
    ampl.param["demand"] = DEMAND
    ampl.solve(solver=solver)

    solution = extract_matrix_solution(ampl.get_variable("Trips"))
    solution["total_cost"] = int(round(ampl.get_value("TotalCost")))
    return solution


In [20]:
variant_result, variant_time = solve_selective_fishing_variant_with_ampl()

print("=== SELECTIVE FISHING VARIANT RESULTS WITH AMPL ===")
print(f"Solution -> {variant_result}")
print(f"Time     -> {variant_time:.8f} seconds")

assert variant_result == VARIANT_EXPECTED
print("The AMPL variant matches the illustrative optimum under the stated assumption.")


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: === SELECTIVE FISHING VARIANT RESULTS WITH AMPL ===
Solution -> {('Atlantic', 'Big Eye'): 4, ('Atlantic', 'Yellowfin'): 2, ('Atlantic', 'Skipjack'): 2, 'total_cost': 80000}
Time     -> 1.57446650 seconds
The AMPL variant matches the illustrative optimum under the stated assumption.


In [21]:
print("=== FINAL COMPARISON ===")
print()
print("BASE PROBLEM")
print(f"Solution -> {base_result}")
print(f"Time     -> {base_time:.8f} seconds")
print()
print("SELECTIVE-FISHING VARIANT")
print(f"Solution -> {variant_result}")
print(f"Time     -> {variant_time:.8f} seconds")
print()


=== FINAL COMPARISON ===

BASE PROBLEM
Solution -> {'pacific_trips': 8, 'atlantic_trips': 3, 'cost': 134000}
Time     -> 1.55548746 seconds

SELECTIVE-FISHING VARIANT
Solution -> {('Atlantic', 'Big Eye'): 4, ('Atlantic', 'Yellowfin'): 2, ('Atlantic', 'Skipjack'): 2, 'total_cost': 80000}
Time     -> 1.57446650 seconds

